# U.S. Electric Vehicle Adoption Analysis
**Data Sources:**
- EV Registrations by State (2023) — U.S. DOE
- U.S. Public EV Charging Infrastructure (2007–2024) — U.S. DOE

**Tools:** Python · Pandas · Matplotlib · Seaborn

Analyzing the state of EV adoption across the U.S. — which states lead, how charging infrastructure has grown, and whether supply keeps pace with adoption.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import openpyxl

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
# Load EV registrations by state
wb = openpyxl.load_workbook('../data/ev_registrations_by_state_2023.xlsx')
ws = wb.active
rows = list(ws.iter_rows(values_only=True))

# Find header row
header_idx = next(i for i, r in enumerate(rows) if r[1] == 'State')
data = [r for r in rows[header_idx+1:] if r[1] and r[2]]
reg_df = pd.DataFrame(data, columns=['_', 'State', 'Registrations'] + [''] * (len(rows[header_idx]) - 3))
reg_df = reg_df[['State', 'Registrations']].dropna()
reg_df['Registrations'] = pd.to_numeric(reg_df['Registrations'], errors='coerce')
reg_df = reg_df.dropna().sort_values('Registrations', ascending=False)

print(f"States: {len(reg_df)}")
print(f"Total EV registrations (2023): {reg_df['Registrations'].sum():,.0f}")
reg_df.head(10)

In [ ]:
# Load charging infrastructure data
wb2 = openpyxl.load_workbook('../data/ev_charging_infrastructure.xlsx')
ws2 = wb2.active
rows2 = list(ws2.iter_rows(values_only=True))

header_idx2 = next(i for i, r in enumerate(rows2) if r[1] == 'Year')
data2 = [r for r in rows2[header_idx2+1:] if r[1] and r[2]]
charge_df = pd.DataFrame(data2, columns=['_', 'Year', 'Charging_Ports', 'Station_Locations'] + [''] * max(0, len(rows2[header_idx2]) - 4))
charge_df = charge_df[['Year', 'Charging_Ports', 'Station_Locations']]
charge_df['Year'] = pd.to_numeric(charge_df['Year'], errors='coerce')
charge_df['Charging_Ports'] = pd.to_numeric(charge_df['Charging_Ports'], errors='coerce')
charge_df['Station_Locations'] = pd.to_numeric(charge_df['Station_Locations'], errors='coerce')
charge_df = charge_df.dropna()

print(charge_df)

In [ ]:
# 1. Top 15 states by EV registrations
top15 = reg_df.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top15['State'][::-1], top15['Registrations'][::-1], color='steelblue', alpha=0.85)
ax.set_title('Top 15 States by EV Registrations (2023)', fontsize=14)
ax.set_xlabel('Number of Registered EVs')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../data/top_states_ev.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2. Charging infrastructure growth over time
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(charge_df['Year'], charge_df['Charging_Ports'], color='steelblue', alpha=0.7, label='Charging Ports')
ax1.set_xlabel('Year')
ax1.set_ylabel('Charging Ports', color='steelblue')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = ax1.twinx()
ax2.plot(charge_df['Year'], charge_df['Station_Locations'], color='darkorange',
         linewidth=2.5, marker='o', label='Station Locations')
ax2.set_ylabel('Station Locations', color='darkorange')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax1.set_title('U.S. Public EV Charging Infrastructure Growth (2007–2024)', fontsize=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../data/charging_infrastructure_growth.png', bbox_inches='tight')
plt.show()

In [ ]:
# 3. EV registration distribution across all states
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(reg_df['State'], reg_df['Registrations'], color='teal', alpha=0.7)
ax.set_title('EV Registrations by State (2023)', fontsize=14)
ax.set_xlabel('State')
ax.set_ylabel('Registrations')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('../data/ev_all_states.png', bbox_inches='tight')
plt.show()

# Top 3 states share
top3_share = reg_df.head(3)['Registrations'].sum() / reg_df['Registrations'].sum() * 100
print(f"Top 3 states hold {top3_share:.1f}% of all EV registrations")
print(f"Latest charging ports (2024): {charge_df['Charging_Ports'].iloc[-1]:,.0f}")
print(f"Latest station locations (2024): {charge_df['Station_Locations'].iloc[-1]:,.0f}")